# 02 · Del CSV a una entrega válida — inicio para completar

Entrenar un clasificador, medirlo y exportar predicciones conservando los identificadores y su orden.

**Ideas que aparecen:** laboratorios 00 y 01; diferencia entre ejemplo y etiqueta. Puedes consultar las cápsulas
opcionales de `ruta/Puentes de entrada.md` cuando alguna te haga falta.

Datos **sintéticos educativos**, creados en este repositorio y dedicados a
CC0-1.0. No representan personas ni un rendimiento oficial IOAI.
Todo el ejercicio usa CPU y archivos locales; no requiere cuentas ni red.

El inicio ya corre. Las celdas de experimentación son lugares para cambiar una idea y observar qué ocurre; la solución está en otro archivo si quieres contrastarla.
Puedes recorrerlo en una o varias sesiones. No hay límite de juez ni
obligación de completar todos los experimentos para abrir el siguiente cuaderno.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATOS = Path("datos")
assert DATOS.is_dir(), "Abre el notebook desde su carpeta: deben existir datos/ e inicio.ipynb."
SEMILLA = 17


## El ciclo completo
Clasifica piezas ficticias: `x1` y `x2` son mediciones disponibles;
`clase` es 0 o 1. Un baseline mayoritario ignora las mediciones.
Una regresión logística aprende una frontera y recibe datos
escalados usando estadísticas de entrenamiento.

La entrega local es `predicciones.csv`, exactamente dos columnas:
`id,clase`. Debe contener todas las filas de transferencia en el
orden de entrada. Las etiquetas de ese conjunto están en un
archivo separado para abrir al finalizar la propuesta. Es una
evaluación educativa visible, no un test privado de competencia.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
datos = pd.read_csv(DATOS / "desarrollo.csv")
train = datos[datos.particion == "train"]
val = datos[datos.particion == "val"]
columnas = ["x1", "x2"]
assert (len(train), len(val)) == (180, 60)
print(train.groupby("clase").size())


## Zona de experimentación
Reemplaza el predictor mayoritario por logística con escalado en
un pipeline. Mantener el split permite atribuir a la propuesta, y no a otras filas,
los cambios que observas entre variantes. Completa la exportación y comprueba que no
ordenas IDs ni predicciones de forma independiente.


In [ ]:
def construir_modelo():
    # TRABAJO: conserva el baseline abajo y mejora esta propuesta.
    return DummyClassifier(strategy="most_frequent")

def exportar(ids, predicciones, destino):
    # Flujo inicial válido; explica por qué estas filas están alineadas.
    pd.DataFrame({"id": np.asarray(ids), "clase": predicciones}).to_csv(destino, index=False)


## Comparación en las mismas condiciones
Baseline mayoritario y propuesta se entrenan con las mismas 180 filas y se evalúan sobre las mismas 60. Mira dónde la propuesta mejora al baseline y dónde todavía se equivoca. El archivo también debe ser válido; una buena métrica no corrige un orden equivocado.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(train[columnas], train.clase)
modelo = construir_modelo().fit(train[columnas], train.clase)
resultado = {
    "baseline": float(accuracy_score(val.clase, baseline.predict(val[columnas]))),
    "validacion": float(accuracy_score(val.clase, modelo.predict(val[columnas]))),
}
print(resultado)


<details><summary>Idea · composición</summary>make_pipeline(StandardScaler(), LogisticRegression(...)) aplica el mismo escalador al predecir.</details>
<details><summary>Idea · filas</summary>predict conserva el orden de sus filas de entrada. Usa los IDs de ese mismo DataFrame.</details>
<details><summary>Idea · control</summary>Lee el CSV exportado y compara su lista completa de IDs con la entrada usando array_equal.</details>


## Transferencia: decide antes de mirar el resultado
Los IDs de la nueva tabla no están ordenados y sus valores no predicen la clase. Congela el modelo, escribe el CSV y recién entonces abre las etiquetas. Usa el mismo modelo, sin ajustar a las respuestas nuevas.


In [ ]:
nuevo = pd.read_csv(DATOS / "transferencia.csv")
pred_nuevo = modelo.predict(nuevo[columnas])
exportar(nuevo.id, pred_nuevo, "predicciones.csv")
envio = pd.read_csv("predicciones.csv")
assert list(envio.columns) == ["id", "clase"]
assert np.array_equal(envio.id, nuevo.id), "El orden de los IDs cambió."
assert len(envio) == 60 and envio.id.is_unique and set(envio.clase) <= {0, 1}
etiquetas = pd.read_csv(DATOS / "etiquetas_transferencia.csv").set_index("id")
y_nuevo = etiquetas.loc[nuevo.id, "clase"].to_numpy()
resultado["transferencia"] = float(accuracy_score(y_nuevo, envio.clase))
resultado["baseline_transferencia"] = float(accuracy_score(y_nuevo, baseline.predict(nuevo[columnas])))
resultado["entrega_valida"] = True


## Mirar la idea
Compara el dibujo con lo que esperabas antes de ejecutar.


In [ ]:
xx, yy = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-30, 30, 100))
grilla = pd.DataFrame({"x1": xx.ravel(), "x2": yy.ravel()})
frontera = modelo.predict(grilla).reshape(xx.shape)
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].contourf(xx, yy, frontera, levels=[-0.5, 0.5, 1.5], cmap="coolwarm", alpha=0.22)
ejes[0].scatter(val.x1, val.x2, c=val.clase, cmap="coolwarm", edgecolor="white")
errores = modelo.predict(val[columnas]) != val.clase.to_numpy()
ejes[0].scatter(val.loc[errores, "x1"], val.loc[errores, "x2"], facecolors="none", edgecolors="black", s=110, label="error")
ejes[0].set(title="La frontera y sus errores en validación", xlabel="x1", ylabel="x2")
ejes[0].legend()
ejes[1].bar(["mayoritario", "propuesta", "transferencia"], [resultado["baseline"], resultado["validacion"], resultado["transferencia"]], color=["#a5adb5", "#287da3", "#65b5af"])
ejes[1].set(title="Misma métrica, conjuntos declarados", ylabel="accuracy", ylim=(0, 1))
fig.tight_layout()
plt.show()


## Para seguir explorando
¿Dónde se equivoca la frontera? Mueve un punto, cambia la escala de una feature o baraja las filas de entrada conservando sus IDs. Distingue un cambio del modelo de un error al construir la entrega.


## Comprobaciones del ejemplo
Estas aserciones detectan errores técnicos en el cuaderno y en su solución de referencia. No son un examen ni una escala de capacidad.


In [ ]:
assert isinstance(resultado, dict)


## Resultado reproducible
Esta celda guarda automáticamente las medidas para comprobar el material. Puedes conservar una copia de tu notebook y tus propias notas; no hay un formulario que rellenar.


In [ ]:
resultado.update({"laboratorio": '02_clasificacion', "version": 'inicio para completar',
                  "datos": "sintéticos CC0-1.0",
                  "metrica": 'accuracy (mayor es mejor)', "split": '180 entrenamiento y 60 validación fijos; 60 transferencia con identificadores desordenados'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
